# Исследование надежности заемщиков


Описание проекта:
Заказчик — кредитный отдел банка. Нужно разобраться, влияет ли семейное положение и количество детей клиента на факт погашения кредита в срок. Входные данные от банка — статистика о платёжеспособности клиентов.

Результаты исследования будут учтены при построении модели кредитного скоринга — специальной системы, которая оценивает способность потенциального заёмщика вернуть кредит банку.

**Описание данных:**

children — количество детей в семье

days_employed — общий трудовой стаж в днях

dob_years — возраст клиента в годах

education — уровень образования клиента

education_id — идентификатор уровня образования

family_status — семейное положение

family_status_id — идентификатор семейного положения

gender — пол клиента

income_type — тип занятости

debt — имел ли задолженность по возврату кредитов

total_income — ежемесячный доход

purpose — цель получения кредита

## Откроем таблицу и изучим общую информацию о данных

In [40]:
import pandas as pd

try:
    data = pd.read_csv('/datasets/data.csv')
except:
    data = pd.read_csv('https://code.s3.yandex.net/datasets/data.csv')

In [41]:
data.head(20)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


In [42]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


## Предобработка данных

### Удаление пропусков

In [43]:
data.isna().sum()

children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2174
purpose                0
dtype: int64

In [44]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['total_income'].isna()), 'total_income'] = \
    data.loc[(data['income_type'] == t), 'total_income'].median()

### Обработка аномальных значений

In [45]:
data['days_employed'] = data['days_employed'].abs()

In [46]:
data.groupby('income_type')['days_employed'].agg('median')

income_type
безработный        366413.652744
в декрете            3296.759962
госслужащий          2689.368353
компаньон            1547.382223
пенсионер          365213.306266
предприниматель       520.848083
сотрудник            1574.202821
студент               578.751554
Name: days_employed, dtype: float64

У двух типов (безработные и пенсионеры) получатся аномально большие значения. Исправить такие значения сложно, поэтому оставим их как есть. Тем более этот столбец не понадобится для исследования.

In [47]:
data['children'].unique()

array([ 1,  0,  3,  2, -1,  4, 20,  5])

In [48]:
data = data[(data['children'] != -1) & (data['children'] != 20)]

In [49]:
data['children'].unique()

array([1, 0, 3, 2, 4, 5])

### Удаление пропусков (продолжение)

In [50]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['days_employed'].isna()), 'days_employed'] = \
    data.loc[(data['income_type'] == t), 'days_employed'].median()

In [51]:
data.isna().sum()

children            0
days_employed       0
dob_years           0
education           0
education_id        0
family_status       0
family_status_id    0
gender              0
income_type         0
debt                0
total_income        0
purpose             0
dtype: int64

### Изменение типов данных

In [52]:
data['total_income'] = data['total_income'].astype(int)

### Обработка дубликатов

In [53]:
data['education'] = data['education'].str.lower()

In [54]:
data.duplicated().sum()

71

In [55]:
data = data.drop_duplicates()

### Категоризация данных

In [56]:
def categorize_income(income):
    try:
        if 0 <= income <= 30000:
            return 'E'
        elif 30001 <= income <= 50000:
            return 'D'
        elif 50001 <= income <= 200000:
            return 'C'
        elif 200001 <= income <= 1000000:
            return 'B'
        elif income >= 1000001:
            return 'A'
    except:
        pass

In [57]:
data['total_income_category'] = data['total_income'].apply(categorize_income)

In [58]:
data['purpose'].unique()

array(['покупка жилья', 'приобретение автомобиля',
       'дополнительное образование', 'сыграть свадьбу',
       'операции с жильем', 'образование', 'на проведение свадьбы',
       'покупка жилья для семьи', 'покупка недвижимости',
       'покупка коммерческой недвижимости', 'покупка жилой недвижимости',
       'строительство собственной недвижимости', 'недвижимость',
       'строительство недвижимости', 'на покупку подержанного автомобиля',
       'на покупку своего автомобиля',
       'операции с коммерческой недвижимостью',
       'строительство жилой недвижимости', 'жилье',
       'операции со своей недвижимостью', 'автомобили',
       'заняться образованием', 'сделка с подержанным автомобилем',
       'получение образования', 'автомобиль', 'свадьба',
       'получение дополнительного образования', 'покупка своего жилья',
       'операции с недвижимостью', 'получение высшего образования',
       'свой автомобиль', 'сделка с автомобилем',
       'профильное образование', 'высшее об

In [59]:
def categorize_purpose(row):
    try:
        if 'автом' in row:
            return 'операции с автомобилем'
        elif 'жил' in row or 'недвиж' in row:
            return 'операции с недвижимостью'
        elif 'свад' in row:
            return 'проведение свадьбы'
        elif 'образов' in row:
            return 'получение образования'
    except:
        return 'нет категории'

In [60]:
data['purpose_category'] = data['purpose'].apply(categorize_purpose)

### Шаг 3. Исследуем и проанализируем данные

#### 3.1 Есть ли зависимость между количеством детей и возвратом кредита в срок?

In [ ]:
# создадим функцию определения категории заёмщика
def children_category(children):
    if 1 <= children <= 2:
        return 'есть дети'
    if children >= 3:
        return 'многодетный'
    return 'нет детей'



In [62]:
# создадим в таблице столбец children_category для анализа и ответа на вопрос
data['children_category'] = data['children'].apply(children_category)

In [63]:
# посмотрим что получилось
data.tail(10)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose,total_income_category,purpose_category,children_category
21515,1,467.685130,28,среднее,1,женат / замужем,0,F,сотрудник,1,109486,заняться образованием,C,получение образования,есть дети
21516,0,914.391429,42,высшее,0,женат / замужем,0,F,компаньон,0,322807,покупка своего жилья,B,операции с недвижимостью,нет детей
21517,0,404.679034,42,высшее,0,гражданский брак,1,F,компаньон,0,178059,на покупку своего автомобиля,C,операции с автомобилем,нет детей
21518,0,373995.710838,59,среднее,1,женат / замужем,0,F,пенсионер,0,153864,сделка с автомобилем,C,операции с автомобилем,нет детей
21519,1,2351.431934,37,ученая степень,4,в разводе,3,M,сотрудник,0,115949,покупка коммерческой недвижимости,C,операции с недвижимостью,есть дети
21520,1,4529.316663,43,среднее,1,гражданский брак,1,F,компаньон,0,224791,операции с жильем,B,операции с недвижимостью,есть дети
21521,0,343937.404131,67,среднее,1,женат / замужем,0,F,пенсионер,0,155999,сделка с автомобилем,C,операции с автомобилем,нет детей
21522,1,2113.346888,38,среднее,1,гражданский брак,1,M,сотрудник,1,89672,недвижимость,C,операции с недвижимостью,есть дети
21523,3,3112.481705,38,среднее,1,женат / замужем,0,M,сотрудник,1,244093,на покупку своего автомобиля,B,операции с автомобилем,многодетный
21524,2,1984.507589,40,среднее,1,женат / замужем,0,F,сотрудник,0,82047,на покупку автомобиля,C,операции с автомобилем,есть дети


In [76]:
# Создаем таблицу методом pivot_table()
debt_children = data.pivot_table(index='children_category', columns='debt', values='gender', aggfunc='count')
# Создаем столбцы
debt_children.columns = ['no_debt', 'debt']
# Считаем долю должников
debt_children['share_of_debtors'] = debt_children['debt'] / (debt_children['debt'] + debt_children['no_debt'])
# Переводим значения в столбце доля дожников в проценты
debt_children['share_of_debtors'] = debt_children['share_of_debtors'].apply('{:.2%}'.format)
# Сортируем по столбцу доля должников по убыванию
debt_children.sort_values(by='share_of_debtors', ascending=False)

,no_debt,debt,share_of_debtors
children_category,,,
есть дети,6222,638,9.30%
многодетный,349,31,8.16%
нет детей,13028,1063,7.54%


**Вывод:** 

Есть зависимость между количеством детей и возвратом кредита в срок - бездетные допускают меньше просрочек по задолженностям. Также есть неточная особенность, что те, у кого от 3 детей допускают меньше просрочек, чем, те у кого меньше 3, но это лишь предположение так как по многодетным данных недостаточно, нужно больше данных для проверки.

#### 3.2 Есть ли зависимость между семейным положением и возвратом кредита в срок?

In [ ]:
# Сгруппируем данные по 'family_status', рассчитаем сумму и количество долгов, вычислим долю должников и отсортируем
debt_from_family_status = pd.DataFrame()
debt_from_family_status['sum_family_status'] = data.groupby('family_status')['debt'].sum()
debt_from_family_status['count_family_status'] = data.groupby('family_status')['debt'].count()
debt_from_family_status['result_family_status'] = debt_from_family_status['sum_family_status'] / debt_from_family_status['count_family_status'] 
debt_from_family_status['result_family_status'] = debt_from_family_status['result_family_status'].apply('{:2%}'.format)
debt_from_family_status.sort_values('result_family_status', ascending = False)


,sum_family_status,count_family_status,result_family_status
family_status,,,
Не женат / не замужем,273,2796,9.763948%
гражданский брак,385,4134,9.313014%
женат / замужем,927,12261,7.560558%
в разводе,84,1189,7.064760%
вдовец / вдова,63,951,6.624606%


**Вывод:** Зависимость есть - люди не в браке и не бывавшие в браке имеют больший процент невозвратов в срок. Но, те кто развелись или овдовели чаще платят в срок, чем люди в браке.

#### 3.3 Есть ли зависимость между уровнем дохода и возвратом кредита в срок?

In [ ]:
# Проанализируем заёмщиков по категориям дохода: подсчет заемщиков, должников и процента должников, отсортируем по убыванию

debt_income_category = data.groupby('total_income_category')['debt'].agg(['count', 'sum', 'mean'])
 
debt_income_category['mean'] = debt_income_category['mean'].apply('{:2%}'.format)
 
debt_income_category = debt_income_category.sort_values(by='mean',ascending=False)
 
debt_income_category = debt_income_category.rename(columns={'count':'Заемщики','sum':'Должники','mean':'Процент должников'})
 
debt_income_category



,Заемщики,Должники,Процент должников
total_income_category,,,
E,22,2,9.090909%
C,15921,1353,8.498210%
A,25,2,8.000000%
B,5014,354,7.060231%
D,349,21,6.017192%


**Вывод:** Будем делать вывод по категоряим B и С, для которых есть достаточная выборка данных. Получается, что с ростом доходов заёмщики реже допускают просрочки по платежам. Или если конкретно, то заёмщики с доходом 200001-1000000 выплачивают кредиты лучше, чем заёмщики с доходом 50001-200000. 

#### 3.4 Как разные цели кредита влияют на его возврат в срок?

In [69]:
# Ваш код будет здесь. Вы можете создавать новые ячейки.
debt_purpose = data.groupby('purpose_category')['debt'].agg(['count', 'sum', 'mean'])

debt_purpose['mean'] = debt_purpose['mean'].apply('{:2%}'.format)
 
debt_purpose = debt_purpose.sort_values(by='mean',ascending=False)
 
debt_purpose = debt_purpose.rename(columns={'count':'Заемщики','sum':'Должники','mean':'Процент должников'})
 
debt_purpose

,Заемщики,Должники,Процент должников
purpose_category,,,
операции с автомобилем,4279,400,9.347978%
получение образования,3988,369,9.252758%
проведение свадьбы,2313,183,7.911803%
операции с недвижимостью,10751,780,7.255139%


**Вывод:** Цели влияют на вероятность возврата кредита в срок: клиенты, берущие кредит на операции с автомобилем или на получение образования, чаще допускают просрочку по выплатам, чем те, кто берут кредит на свадьбу или на операции с недвижимостью. 

#### 3.5 Какие могут быть возможные причины появления пропусков в исходных данных?

Причинами могут быть: неверный ввод данных, сокрытие информации, повреждение файла данных при сохранении, технические проблемы.

#### 3.6 Поясним почему заполнить пропуски медианным значением — лучшее решение для количественных переменных.

Среднее значение некорректно характеризует данные в том случае, когда некоторые количественные значения сильно отличаются от большинства других. Медиана даёт более правдоподобную картину, т.к. менее чувствительна к выбросам.

### Шаг 4: общий вывод.

   Были изучены данные о заемщиках банка с целью выявления недобросовестных заемщиков.

   Прежде всего, данные были подготовлены для анализа.
    
   Были обнаружены пропущенные значения в столбцах "days_employed" и "total_income". Поскольку доход ("total_income") зависит от типа занятости, заемщики были сгруппированы по типам занятости, в каждой группе был рассчитан медианный доход. Пропуски в столбце "total income" были заполнены медианным доходом, рассчитанным для той группы занятости, к которой принадлежит заемщик. Медианные значения для заполнения пропусков были выбраны потому, что медиана, описывающая показатели в группе, менее чувствительна к аномальным значениям в рассматриваемой выборке. 
    
   Также были выявлены аномальные отрицательные значения в столбце "days_employed", они были заменены на положительные. Данные с аномальными значениями в столбце "children" были удалены. 
    
   Следует помнить, что причинами пропусков и аномальных значений могут быть: неверный ввод данных, сокрытие информации, повреждение файла данных при сохранении, технические проблемы.
    
   В процессе анализа заемщики были разделены на 5 категорий по уровню дохода. Были изучены цели получения кредита, они были разделены на 4 категории: «операции с автомобилем», «операции с недвижимостью», «проведение свадьбы», «получение образования».
   
   В результате анализа характеристик, описывающих заемщиков, удалось предположить следующие закономерности. 
    
   Заёмщики без детей реже допускают задолженность по выплатам (7.5%), чем заёмщики с детьми (9.3%). По многодетным заёмщикам (от трёх детей и более) данных пока мало, и можно сделать только предположительный вывод, что они допускают задолженность не чаще, чем заёмщики с менее чем тремя детьми.
    
   Семейное положение влияет на возврат кредита в срок: заёмщики со статусами "не женат / не замужем", а также "гражданский брак" чаще имеют задолженность, чем клиенты из категорий "женат / замужем", "в разводе" и "вдовец / вдова". Мы предполагаем, что более информативным может быть исследование связи невыплат кредита и возраста заемщиков. 
    
   Для лучшего выявления зависимости уровня дохода и невыплат кредитов желательно получить больше данных клиентов с малыми доходами (0 – 30 000, 30 001 – 50 000) и высоким уровнем заработка (1 000 001 и выше), тогда как данных о средних категориях 
(50 001 – 200 000 и 200 001 – 1 000 000) достаточно. Исходя из этих данных, можно выявить закономерность, что с ростом доходов заёмщики реже допускают просрочки по платежам (заёмщики с доходом 200 001 - 1 000 000 выплачивают кредиты лучше, чем заёмщики с доходом 50 001 - 200 000).
    
   Цели кредита влияют на возврат в срок. Клиенты, оформившие кредит на операции с недвижимостью, лучше всех возвращают кредит (7,26% невозврата кредита в срок). Заемщики, оформившие кредит на операции с автомобилем, наименее надёжны (9,35% невозврата кредита в срок). Между ними находятся: те, кто брал кредит на проведение свадьбы (7,91%) и получение образования (9,25%).
    
   В качестве желательных улучшений можно порекомендовать улучшить техническое обеспечение сбора данных, чтобы получать меньше ошибочных значений и пропусков, также сделать более обширную по значимым признакам выборку: например, увеличить количество данных о заемщиках с малыми и очень большими доходами, получить больше данных о многодетных заемщиках. 



